In [29]:
%pip install pandas requests --quiet

Note: you may need to restart the kernel to use updated packages.


In [30]:
import io, requests
import pandas as pd

INPUT_PATH = "locations-raw.csv"      # change if needed
OUTPUT_PATH = "locations-geocoded.csv"

df = pd.read_csv(INPUT_PATH).reset_index(drop=True)

In [31]:
# Build the headerless batch payload: id, street, city, state, zip
batch = pd.DataFrame({
    "id": df.index,
    "street": df["Site"].str.strip(),
    "city": "San Francisco",
    "state": "CA",
    "zip": "",
})
csv_buf = io.StringIO()
batch.to_csv(csv_buf, header=False, index=False)

In [32]:
# One request geocodes all rows (free, no key, US-only)
resp = requests.post(
    "https://geocoding.geo.census.gov/geocoder/locations/addressbatch",
    files={"addressFile": ("addresses.csv", csv_buf.getvalue())},
    data={"benchmark": "Public_AR_Current"},
    timeout=180,
)
resp.raise_for_status()

In [33]:
cols = ["id", "input_address", "match", "match_type",
        "matched_address", "coordinates", "tiger_line_id", "side"]
geo = pd.read_csv(io.StringIO(resp.text), header=None, names=cols,
                  dtype={"id": int}).sort_values("id").reset_index(drop=True)

coords = geo["coordinates"].str.split(",", expand=True)
geo["longitude"] = pd.to_numeric(coords[0], errors="coerce")
geo["latitude"]  = pd.to_numeric(coords[1], errors="coerce")

result = df.join(geo.set_index("id")[["match", "matched_address", "longitude", "latitude"]])
result["zip"] = result["matched_address"].str.extract(r"(\d{5})(?:-\d{4})?\s*$")

In [34]:
# manually deal with rows with NaN
manual_fixes = {
    "1485 Bay Shore Blvd": "1485 Bayshore Blvd, San Francisco, CA 94124",
    "5th St & Brannan St": "5th St & Brannan St, San Francisco, CA 94107",
    "1201 08th St":        "1201 8th St, San Francisco, CA 94107",
}

for site, full in manual_fixes.items():
    mask = result["Site"] == site
    result.loc[mask, "matched_address"] = full
    result.loc[mask, "zip"] = pd.Series([full]).str.extract(r"(\d{5})(?:-\d{4})?\s*$").iloc[0, 0]
    result.loc[mask, "match"] = "Match (manual)"

result[result["Site"].isin(manual_fixes)][["Site", "matched_address", "zip", "match"]]

,Site,matched_address,zip,match
3,1201 08th St,"1201 8th St, San Francisco, CA 94107",94107,Match (manual)


In [35]:
# Geocode the 3 fixed addresses to fill in coordinates (Census single-line, keyless)
def geocode_oneline(address):
    r = requests.get(
        "https://geocoding.geo.census.gov/geocoder/locations/onelineaddress",
        params={"address": address, "benchmark": "Public_AR_Current", "format": "json"},
        timeout=60,
    )
    r.raise_for_status()
    matches = r.json()["result"]["addressMatches"]
    if matches:
        c = matches[0]["coordinates"]
        return c["y"], c["x"]          # (latitude, longitude)
    return None, None

for site, full in manual_fixes.items():
    lat, lon = geocode_oneline(full)
    mask = result["Site"] == site
    result.loc[mask, "latitude"] = lat
    result.loc[mask, "longitude"] = lon
    print(f"{site:<22} -> {lat}, {lon}")

# add coordinate for the intersection (Census doesn't geocode intersections)
mask = result["Site"] == "5th St & Brannan St"
result.loc[mask, ["latitude", "longitude"]] = [37.776634, -122.398814]

result[result["Site"].isin(manual_fixes)][["Site", "matched_address", "latitude", "longitude"]]

1485 Bay Shore Blvd    -> 37.725535736384, -122.401401989518
5th St & Brannan St    -> None, None
1201 08th St           -> 37.766696846569, -122.399573812702


,Site,matched_address,latitude,longitude
3,1201 08th St,"1201 8th St, San Francisco, CA 94107",37.766697,-122.399574


In [36]:
# Rows that still did not match (fix source address or geocode manually)
result[result["match"] != "Match"][["Site", "match"]]

,Site,match
3,1201 08th St,Match (manual)
10,5th and Brannan,Tie
28,640 Chavez,No_Match


In [37]:
# drop the match column, then save
final = result.drop(columns=["match"])
final.to_csv(OUTPUT_PATH, index=False)
final[["Site", "matched_address", "zip", "latitude", "longitude"]]

,Site,matched_address,zip,latitude,longitude
0,50 Quint St,"50 QUINT ST, SAN FRANCISCO, CA, 94124",94124,37.746304,-122.388371
1,1101-1123 Sutter St,"1123 SUTTER ST, SAN FRANCISCO, CA, 94109",94109,37.787854,-122.418863
2,2293 Powell St,"2293 POWELL ST, SAN FRANCISCO, CA, 94133",94133,37.805731,-122.412003
3,1201 08th St,"1201 8th St, San Francisco, CA 94107",94107,37.766697,-122.399574
4,2270 McKinnon Ave,"2270 MCKINNON AVE, SAN FRANCISCO, CA, 94124",94124,37.742872,-122.401479
5,130 Townsend Street,"130 TOWNSEND ST, SAN FRANCISCO, CA, 94107",94107,37.779968,-122.391441
6,2860 16th St,"2860 16TH ST, SAN FRANCISCO, CA, 94103",94103,37.765272,-122.416944
7,1428 Yosemite Ave,"1428 YOSEMITE AVE, SAN FRANCISCO, CA, 94124",94124,37.725389,-122.388969
8,1111 Pennsylvania Avenue,"1111 PENNSYLVANIA AVE, SAN FRANCISCO, CA, 94107",94107,37.752470,-122.392558
9,2450 Alameda Street,"2450 ALAMEDA ST, SAN FRANCISCO, CA, 94103",94103,37.768327,-122.409236


In [38]:
# Manually fill in the two rows the batch geocoder couldn't match
manual = {
    "640 Chavez": {
        "matched_address": "640 CESAR CHAVEZ ST, SAN FRANCISCO, CA 94124",
        "zip": "94124",
        "latitude": 37.750961,
        "longitude": -122.384315,
    },
    "5th and Brannan": {
        "matched_address": "5th St & Brannan St, SAN FRANCISCO, CA 94107",
        "zip": "94107",
        "latitude": 37.776653,
        "longitude": -122.398819,
    },
}

for site, vals in manual.items():
    mask = final["Site"].str.strip() == site      # .strip() handles the "640 Chavez " trailing space
    if not mask.any():
        print(f"WARNING: no row matched {site!r} — check the exact Site spelling")
    for col, val in vals.items():
        final.loc[mask, col] = val

final.to_csv(OUTPUT_PATH, index=False)             # re-save so the CSV includes these fixes
final[final["Site"].str.strip().isin(manual)][["Site", "matched_address", "zip", "latitude", "longitude"]]

,Site,matched_address,zip,latitude,longitude
10,5th and Brannan,"5th St & Brannan St, SAN FRANCISCO, CA 94107",94107,37.776653,-122.398819
28,640 Chavez,"640 CESAR CHAVEZ ST, SAN FRANCISCO, CA 94124",94124,37.750961,-122.384315


In [39]:
final.to_csv(OUTPUT_PATH, index=False)
final[["Site", "matched_address", "zip", "latitude", "longitude"]]

,Site,matched_address,zip,latitude,longitude
0,50 Quint St,"50 QUINT ST, SAN FRANCISCO, CA, 94124",94124,37.746304,-122.388371
1,1101-1123 Sutter St,"1123 SUTTER ST, SAN FRANCISCO, CA, 94109",94109,37.787854,-122.418863
2,2293 Powell St,"2293 POWELL ST, SAN FRANCISCO, CA, 94133",94133,37.805731,-122.412003
3,1201 08th St,"1201 8th St, San Francisco, CA 94107",94107,37.766697,-122.399574
4,2270 McKinnon Ave,"2270 MCKINNON AVE, SAN FRANCISCO, CA, 94124",94124,37.742872,-122.401479
5,130 Townsend Street,"130 TOWNSEND ST, SAN FRANCISCO, CA, 94107",94107,37.779968,-122.391441
6,2860 16th St,"2860 16TH ST, SAN FRANCISCO, CA, 94103",94103,37.765272,-122.416944
7,1428 Yosemite Ave,"1428 YOSEMITE AVE, SAN FRANCISCO, CA, 94124",94124,37.725389,-122.388969
8,1111 Pennsylvania Avenue,"1111 PENNSYLVANIA AVE, SAN FRANCISCO, CA, 94107",94107,37.752470,-122.392558
9,2450 Alameda Street,"2450 ALAMEDA ST, SAN FRANCISCO, CA, 94103",94103,37.768327,-122.409236
